# 05 maps: the current state with the assignments

Draws the tables `04_assigned_coverage.sql` builds into two PDFs on the stage `@td_maps`, inside Snowflake, in the look of
`tools/maps/zip_pages.py` (owner, 2026-11-17: "I want the look and feel of those maps"): ZCTA fills, district outlines,
dashed state lines, grey land in no ZCTA, the NYC and BOS-WAS insets, the major metros named.
Nothing here changes a territory (owner, 2026-11-16): it only draws `TD_MAP_PIECE`, `TD_MAP_CELL`, `TD_MAP_ZIP` and `TD_MAP_WHOLESALER`.

**Once, before the first run:** upload two files from the repo to the stage root `@td_maps`: `tools/maps/look.py` and
`reference/2025/areas.csv.gz`. Run the Setup cell first (it creates the stage), then in Snowsight: Catalog, Database Explorer,
`LR_STRATEGY`, Stages, `TD_MAPS`, `+ Files`. The Setup cell stops with a message while either file is missing; upload a new
`look.py` there whenever the repo's changes.

**Before the first run:** a warehouse-runtime notebook in `ssdb_integration.lr_strategy`, with the packages `geopandas` and
`matplotlib` added (Packages menu, Anaconda channel). Run `03_assignments.sql`, then `04_assigned_coverage.sql`.

**Run:** Run all.  The PDFs land in `@td_maps/<run time>/`: Catalog, Database Explorer, `LR_STRATEGY`, Stages, `TD_MAPS`, then download.

- `td_maps_territories_and_overview.pdf`: first one page per firm and territory channel (WH, FI): today's territories, each kept
  wholesaler a numbered district (`NN Name` in the legend), ZIPs of a wholesaler not kept hatched grey; then the overview: per role,
  the kept holders on each ZIP (overlap); per estimate, where opportunity is open or has no holder; per estimate, the main map's
  opportunity by ZIP; per firm, main-map sales by ZIP.
- `td_maps_wholesalers.pdf`: one page per kept wholesaler with a role: their pieces in role (sole or shared with another kept
  holder of the role) and out of role, their assigned states outlined, and their numbers.

The PDFs hold wholesaler names and sales: they stay with the owner.  Alaska, Hawaii and the territories are left out.
ZIPs with no ZCTA shape (about 1.3% of the opportunity) are in the numbers but cannot be drawn.  A ZIP held by two wholesalers of
one firm in one channel is drawn in the first kept holder's colour (by name) and counted in the page's footer.

In [ ]:
# Setup: the only cell to edit
import importlib
import os
import sys
import tempfile

from snowflake.snowpark.context import get_active_session

session = get_active_session()
DB = "ssdb_integration.lr_strategy"
UNION_TBL = f"{DB}.sales_opp_2026_11_9"
ZCTA_TBL = f"{DB}.census_zip_codes_zcta"
STATE_TBL = f"{DB}.census_states"
CBSA_TBL = f"{DB}.census_metro_areas_cbsa"
STAGE = f"@{DB}.td_maps"

# Nothing below is edited.  look.py and areas.csv.gz are uploaded once to the stage root (the markdown above).
session.sql(f"CREATE STAGE IF NOT EXISTS {DB}.td_maps DIRECTORY = (ENABLE = TRUE)").collect()
HERE = tempfile.mkdtemp()
for fname in ("look.py", "areas.csv.gz"):
    try:
        session.file.get(f"{STAGE}/{fname}", HERE)
    except Exception as e:
        raise RuntimeError(f"{STAGE}/{fname} is not readable ({e}); upload tools/maps/look.py and "
                           f"reference/2025/areas.csv.gz to {STAGE}, then run this cell again") from e
    if not os.path.exists(os.path.join(HERE, fname)):
        raise RuntimeError(f"{STAGE}/{fname} is missing; upload tools/maps/look.py and "
                           f"reference/2025/areas.csv.gz to {STAGE}, then run this cell again")
AREAS = os.path.join(HERE, "areas.csv.gz")
sys.path.insert(0, HERE)
importlib.invalidate_caches()
import look
look = importlib.reload(look)   # a re-uploaded look.py is picked up without a kernel restart

In [ ]:
# Read the tables (no territory is changed: these are the 04 tables as they are)
from types import SimpleNamespace

import geopandas as gpd
import pandas as pd
import shapely
from shapely import wkb

NOT_CONUS = ("AK", "HI", "PR", "GU", "VI", "AS", "MP")
STATE_ABBR = {"01": "AL", "04": "AZ", "05": "AR", "06": "CA", "08": "CO", "09": "CT", "10": "DE", "11": "DC", "12": "FL",
              "13": "GA", "16": "ID", "17": "IL", "18": "IN", "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME",
              "24": "MD", "25": "MA", "26": "MI", "27": "MN", "28": "MS", "29": "MO", "30": "MT", "31": "NE", "32": "NV",
              "33": "NH", "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND", "39": "OH", "40": "OK", "41": "OR",
              "42": "PA", "44": "RI", "45": "SC", "46": "SD", "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA",
              "53": "WA", "54": "WV", "55": "WI", "56": "WY"}
SIMPLIFY_M = 250.0   # td/output.py's SIMPLIFY_M: the ZIP outlines are simplified exactly as its zcta_polygons does


def q(sql):
    return session.sql(sql).to_pandas()


def projected(df):
    """The WKB column as shapely geometries in EPSG:5070 (td's geo.CRS)."""
    return list(gpd.GeoSeries([wkb.loads(bytes(b)) for b in df["WKB"]], crs="EPSG:4326").to_crs(epsg=5070))


nc = ", ".join(f"'{s}'" for s in NOT_CONUS)
MAPPED = f"""(SELECT zip_code FROM {DB}.td_map_cell WHERE COALESCE(state, '') NOT IN ({nc})
              UNION SELECT zip_code FROM {DB}.td_map_piece WHERE COALESCE(state, '') NOT IN ({nc}))"""

# census states, cartographic 2025: the CONUS states, by FIPS
d = q(f"SELECT RIGHT(geo_id, 2) AS fips, ST_ASWKB(geo_boundary) AS wkb FROM {STATE_TBL}")
st = {STATE_ABBR[f]: g for f, g in zip(d["FIPS"], projected(d)) if f in STATE_ABBR}
land = shapely.union_all(list(st.values()))
shapely.prepare(land)

# every ZCTA at full detail, ten queries by first digit; simplified in shapely as td/output.py does.  The ZCTAs of the
# map tables are drawn; the others only count as covered, so land they cover is not drawn as land in no ZCTA.
polys, others = {}, {}
for digit in "0123456789":
    d = q(f"""
        SELECT RIGHT(z.geo_id, 5) AS zip_code, ST_ASWKB(z.geo_boundary) AS wkb, k.zip_code IS NOT NULL AS mapped
        FROM {ZCTA_TBL} z LEFT JOIN (SELECT DISTINCT zip_code FROM {MAPPED}) k ON k.zip_code = RIGHT(z.geo_id, 5)
        WHERE LEFT(RIGHT(z.geo_id, 5), 1) = '{digit}'""")
    for z, g, m in zip(d["ZIP_CODE"], projected(d), d["MAPPED"]):
        g = g.simplify(SIMPLIFY_M, preserve_topology=True)
        if m:
            polys[z] = g
        elif land.intersects(g.representative_point()):
            others[z] = g
gap = look.no_zcta_land({**polys, **others}, st)
areas = look.read_areas(AREAS)

zip_cbsa = q(f"""
    SELECT z.zip_code, c.cbsa
    FROM (SELECT RIGHT(z.geo_id, 5) AS zip_code, ST_CENTROID(z.geo_boundary) AS ctr
          FROM {ZCTA_TBL} z JOIN (SELECT DISTINCT zip_code FROM {MAPPED}) k ON k.zip_code = RIGHT(z.geo_id, 5)) z
    JOIN (SELECT RIGHT(geo_id, 5) AS cbsa, geo_boundary FROM {CBSA_TBL} WHERE geo_name LIKE '%Metro Area') c
      ON ST_INTERSECTS(z.ctr, c.geo_boundary)""").drop_duplicates("ZIP_CODE")
zip_role = q(f"SELECT zip_code, role, n_role_holders, n_role_firms FROM {DB}.td_map_zip")
zip_cell = q(f"""
    SELECT est_source, zip_code, SUM(mm_opp) AS opp,
           SUM(CASE WHEN holder_status = 'KEPT IN ROLE' THEN mm_opp ELSE 0 END) AS opp_in_role,
           SUM(CASE WHEN holder_status = 'KEPT OUT OF ROLE' THEN mm_opp ELSE 0 END) AS opp_out_of_role,
           SUM(CASE WHEN holder_status = 'NOT KEPT (OPEN)' THEN mm_opp ELSE 0 END) AS opp_open,
           SUM(CASE WHEN holder_status = 'NO HOLDER TODAY' THEN mm_opp ELSE 0 END) AS opp_no_holder
    FROM {DB}.td_map_cell GROUP BY est_source, zip_code""")
zip_sales = q(f"""
    SELECT source, LPAD(LEFT(TRIM(zip_code), 5), 5, '0') AS zip_code, SUM(actual_sales_amount) AS sales
    FROM {UNION_TBL}
    WHERE UPPER(TRIM(channel)) IN ('NATIONAL', 'WH', 'FI') AND actual_sales_amount <> 0
    GROUP BY source, LPAD(LEFT(TRIM(zip_code), 5), 5, '0')""")
terr = q(f"""
    SELECT DISTINCT source, wholesaler, kept, zip_code, terr_channel
    FROM {DB}.td_map_piece WHERE terr_channel IN ('WH', 'FI')""")
pieces = q(f"""
    SELECT source, wholesaler, role, zip_code, state, terr_channel, piece_in_role, in_assigned_state, n_role_holders
    FROM {DB}.td_map_piece WHERE kept = 'Y' AND role IN ('NATIONAL', 'WH', 'FI', 'WIFI')""")
wsum = q(f"SELECT * FROM {DB}.td_map_wholesaler WHERE kept = 'Y' AND role IN ('NATIONAL', 'WH', 'FI', 'WIFI')")
print(len(polys), "ZIP shapes,", len(others), "more ZCTAs covering land,", len(st), "states,", len(terr), "territory rows,",
      len(pieces), "pieces,", wsum[["SOURCE", "WHOLESALER"]].drop_duplicates().shape[0], "kept wholesalers with a role")

In [ ]:
# Drawing: functions of the tables above only (look.py, from tools/maps, holds the shared look)
import collections
import io

import matplotlib
import numpy as np
import pandas as pd
import shapely

matplotlib.use("Agg")
import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.collections import LineCollection, PatchCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, PathPatch, Rectangle
from pyproj import Transformer

DPI = 130
LAND = "#f6f6f6"            # the ZIPs and land the page does not colour
OTHER_EDGE = "#b8b8b8"      # zip_pages' hatch for ZIPs of another planning channel; here, ZIPs of a wholesaler not kept
NO_ZCTA_TEXT = "Land in no ZCTA: unassigned, part of no district"
OPEN_TEXT = "open (holder not kept)"
OTHERS = "#c9d3df"           # a wholesaler page: the other kept holders of the role (not the grey of land in no ZCTA)
HOLDER_BANDS = {"1": 1, "2": 2, "3": 4, "4+": 6}    # kept holders on the ZIP -> a step of the clean Blues ramp
STATUS_COLOURS = {"open (holder not kept)": "#d7301f", "no holder today": "#fc8d59", "kept, out of role": "#fdcc8a",
                  "kept, in role": "#e0ecf4"}
FIFTHS = (0.2, 0.85)        # the part of a sequential ramp the fifths use
METRO_HANDLE = Line2D([], [], marker="o", linestyle="none", markersize=4, markerfacecolor="#111111",
                      markeredgecolor="white", label="Major metro (CBSA)")


def money(x):
    # a bare $ pair starts matplotlib mathtext: escape it
    return f"\\${x / 1e6:,.1f}M" if abs(x) < 1e9 else f"\\${x / 1e9:,.2f}B"


def fill(ax, geoms, box, **kw):
    """The polygons of `geoms` that reach `box`, as one collection."""
    geoms = [g for g in geoms if g is not None and not g.is_empty]
    if box is not None and geoms:
        b = shapely.bounds(np.array(geoms, dtype=object))
        hit = (b[:, 2] >= box[0]) & (b[:, 0] <= box[2]) & (b[:, 3] >= box[1]) & (b[:, 1] <= box[3])
        geoms = [g for g, k in zip(geoms, hit) if k]
    if geoms:
        ax.add_collection(PatchCollection([PathPatch(look.polygon_path(g)) for g in geoms], **kw))


def frame_axes(ax, ctx, paint, box, fs):
    """One map: land, land in no ZCTA, the page's own layers (`paint`), dashed state lines."""
    fill(ax, list(ctx.st.values()), box, facecolor=LAND, edgecolor="none", zorder=0)
    fill(ax, [ctx.gap], box, facecolor=look.UNASSIGNED, edgecolor="none", zorder=1)
    paint(ax, box, fs)
    fill(ax, list(ctx.st.values()), box, facecolor="none", edgecolor="#606060", linewidth=0.5, linestyle=(0, (4, 2)), zorder=3)
    ax.set_xlim(box[0], box[2])
    ax.set_ylim(box[1], box[3])
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def fit(box, ratio):
    """`box` widened (about its centre) to the width / height `ratio` of the axes it fills."""
    x0, y0, x1, y1 = box
    w, h = x1 - x0, y1 - y0
    if w / h < ratio:
        x0, x1 = (x0 + x1) / 2 - h * ratio / 2, (x0 + x1) / 2 + h * ratio / 2
    else:
        y0, y1 = (y0 + y1) / 2 - w / ratio / 2, (y0 + y1) / 2 + w / ratio / 2
    return x0, y0, x1, y1


def render(title, ctx, paint, handles, metro, captions, box=None, insets=None, side="", legend_fs=6.3):
    """A page in zip_pages' frame: the map left (CONUS, or `box`), the insets right, the legend, the title
    and three caption lines at the bottom.  `paint(ax, box, fs)` draws the page's own layers."""
    insets = [look.NYC, look.CORRIDOR] if insets is None else insets
    height, y, h = look.frame(True)
    fig = plt.figure(figsize=(20, height), dpi=DPI, facecolor="white")
    ax = fig.add_axes([0.005, y(0.12), 0.66, h(0.80)])
    if box is None:
        x0, y0, x1, y1 = shapely.total_bounds(list(ctx.st.values()))
        pad = 0.01 * (x1 - x0)
        box = (x0 - pad, y0 - pad, x1 + pad, y1 + pad)
    else:
        box = fit(box, 0.66 * 20 / (h(0.80) * height))
    frame_axes(ax, ctx, paint, box, 7)
    look.label_metros(ax, metro)
    hgt = h(0.80) / max(len(insets), 1)
    for i, (name, lonlat) in enumerate(insets):
        b = look.bbox(lonlat)
        ax.add_patch(Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, edgecolor="black", linewidth=1.0, zorder=7))
        ax.text(b[0], b[3], f" {chr(65 + i)}", ha="left", va="bottom", fontsize=9, fontweight="bold", zorder=7)
        iax = fig.add_axes([0.665, y(0.12) + h(0.80) - (i + 1) * hgt + h(0.01), 0.20, hgt - h(0.02)])
        frame_axes(iax, ctx, paint, b, 8)
        iax.set_title(f"{chr(65 + i)}: {name}", fontsize=10)
        look.label_metros(iax, metro, fs=7.5)
    handles = list(handles) + [Patch(facecolor=look.UNASSIGNED, edgecolor="none", label=NO_ZCTA_TEXT)]
    if metro:
        handles.append(METRO_HANDLE)
    left = 0.868 if insets else 0.68
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(left, y(0.93)), fontsize=legend_fs, frameon=False, ncol=1,
               handlelength=1.2, borderaxespad=0)
    if side:
        fig.text(0.68, y(0.74), side, fontsize=9, va="top", family="monospace")
    fig.suptitle(title, fontsize=17, fontweight="bold", y=y(0.975))
    for s, yy, size, colour in zip(captions, (0.075, 0.045, 0.022), (10.5, 8.5, 8.5), ("black", "#404040", "#404040")):
        fig.text(0.01, yy, s, fontsize=size, color=colour)
    return fig


def metro_weights(zip_cell, zip_cbsa):
    """{estimate: {cbsa: main-map opportunity $}} from each ZIP's CBSA Metro Area."""
    m = zip_cell.merge(zip_cbsa, on="ZIP_CODE")
    return {est: g.groupby("CBSA")["OPP"].sum().to_dict() for est, g in m.groupby("EST_SOURCE")}


def metro_list(ctx, ests, n=None):
    """[(x, y, name)] in EPSG:5070 of the top metros by the opportunity of `ests` (their mean when several: the
    estimates are never added)."""
    tot = collections.Counter()
    for e in ests:
        for c, v in ctx.weights.get(e, {}).items():
            tot[c] += v / len(ests)
    top = look.top_metros(dict(tot), ctx.areas, **({"n": n} if n else {}))
    if not top:
        return []
    xs, ys = Transformer.from_crs("EPSG:4326", "EPSG:5070", always_xy=True).transform([t[1] for t in top], [t[2] for t in top])
    return [(x, y, t[0]) for t, x, y in zip(top, xs, ys)]


def note(ests):
    return (f"Metros: the top CBSA Metro Areas by main-map opportunity \\$ ({'/'.join(sorted(ests))} estimate), as room allows, at "
            f"their internal points (2025 gazetteer).  Land in no ZCTA: census states less every CONUS ZCTA (250 m simplified), "
            f"opened by {look.OPEN_M / 1000:g} km.")


def territory_page(src, ch, t, ctx):
    """Today's territories of firm `src` in territory channel `ch`: each kept wholesaler a district."""
    polys = ctx.polys
    t = t.drop_duplicates(["ZIP_CODE", "WHOLESALER"])
    kept = sorted(t.loc[t["KEPT"] == "Y", "WHOLESALER"].unique(), key=lambda n: (n.upper(), n))
    num = {n: f"{i + 1:02d}" for i, n in enumerate(kept)}
    held = t.assign(up=t["WHOLESALER"].str.upper(), nk=t["KEPT"] != "Y").sort_values(["ZIP_CODE", "nk", "up", "WHOLESALER"])
    n_two = int((held.groupby("ZIP_CODE")["WHOLESALER"].nunique() > 1).sum())
    first = held.drop_duplicates("ZIP_CODE")           # the first holder: kept ones first, then by name
    rows, open_zips = {n: [] for n in kept}, []
    for z, w, nk in zip(first["ZIP_CODE"], first["WHOLESALER"], first["nk"]):
        (open_zips if nk else rows[w]).append(z)
    unions = {n: shapely.union_all([polys[z] for z in zs if z in polys]).buffer(150).buffer(-150) for n, zs in rows.items()}
    near = look.neighbours(unions)
    colour = look.colouring_far(unions, near, look.CLEAN_PALETTE)
    lines = [shapely.line_merge(g) if g.geom_type == "MultiLineString" else g for g in look.borders(unions, near)]
    segs = [np.asarray(ln.coords)[:, :2] for g in lines for ln in shapely.get_parts(g)
            if ln.geom_type == "LineString" and not ln.is_empty]
    shapes = [polys[z] for z in open_zips if z in polys]
    other = shapely.union_all(shapes).buffer(150).buffer(-150) if shapes else None

    def paint(ax, box, fs):
        if other is not None:
            fill(ax, [other], box, facecolor=LAND, edgecolor=OTHER_EDGE, hatch="////", linewidth=0, zorder=1)
        for n, zs in rows.items():
            fill(ax, [polys[z] for z in zs if z in polys], box, facecolor=colour[n], edgecolor=colour[n], linewidth=0.25,
                 alpha=0.75, zorder=2)
        ax.add_collection(LineCollection(segs, colors="#202020", linewidths=0.8, zorder=4))
        view = shapely.box(*box)
        for n, u in unions.items():
            g = u.intersection(view)
            if g.is_empty:
                continue
            p = max(getattr(g, "geoms", [g]), key=lambda part: part.area).representative_point()
            ax.text(p.x, p.y, num[n], ha="center", va="center", fontsize=fs, fontweight="bold", zorder=6,
                    path_effects=[pe.withStroke(linewidth=2.2, foreground="white")])

    k = len(kept)
    handles = [Patch(facecolor=colour[n], alpha=0.75, edgecolor="#202020", label=f"{num[n]} {n[:30]}") for n in kept]
    if open_zips:
        handles.append(Patch(facecolor=LAND, edgecolor=OTHER_EDGE, hatch="////", label=OPEN_TEXT))
    missing = len({z for zs in list(rows.values()) + [open_zips] for z in zs} - set(polys))
    c1 = (f"{len(first):,} ZIPs: {len(first) - len(open_zips):,} held by the {k} kept wholesalers, {len(open_zips):,} only by a "
          f"wholesaler not kept.  {n_two:,} ZIPs with 2 holders in {src} {ch}: drawn in the first holder's colour (kept holders first, "
          f"then by name)" + (f"  ·  {missing:,} ZIPs with no shape" if missing else ""))
    c2 = ("Today's territories as they are, one district per kept wholesaler; every ZIP held is filled.  Grey: land in no ZCTA, "
          "unassigned, not a hole in a district.  Black: borders between districts; dashed: state lines.")
    return render(f"{src} {ch} territories today: {k} kept wholesalers", ctx, paint, handles, metro_list(ctx, [src]),
                  [c1, c2, note([src])], legend_fs=6.3 if k < 30 else 5.6)


def territory_pages(terr, ctx):
    for src in sorted(terr["SOURCE"].unique()):
        for ch in ("WH", "FI"):
            t = terr[(terr["SOURCE"] == src) & (terr["TERR_CHANNEL"] == ch)]
            if len(t):
                yield territory_page(src, ch, t, ctx)


def category_page(title, cats, colours, ctx, ests, captions):
    """Each of `cats` ({label: ZIPs}) filled in its colour."""
    drawn = {k: [ctx.polys[z] for z in cats.get(k, []) if z in ctx.polys] for k in colours}

    def paint(ax, box, fs):
        for k, c in colours.items():
            fill(ax, drawn[k], box, facecolor=c, edgecolor=c, linewidth=0.25, zorder=2)

    handles = [Patch(facecolor=c, edgecolor="#999", label=f"{k} ({len(drawn[k]):,})") for k, c in colours.items()]
    return render(title, ctx, paint, handles, metro_list(ctx, ests), list(captions) + [note(ests)], legend_fs=6.5)


def fifths(df, column, cmap, polys):
    """({label: ZIPs}, {label: colour}): the ZIPs with `column` above 0 and a shape, in fifths of the ZIPs."""
    g = df[(df[column] > 0) & df["ZIP_CODE"].isin(list(polys))]
    cats, colours = {}, {}
    if len(g):
        bins = pd.qcut(g[column].rank(method="first"), q=min(5, len(g)), labels=False)
        for b in sorted(bins.unique()):
            part = g[bins == b]
            label = f"{money(part[column].min())} to {money(part[column].max())}"
            cats[label] = list(part["ZIP_CODE"])
            colours[label] = matplotlib.colors.to_hex(plt.get_cmap(cmap)(FIFTHS[0] + (FIFTHS[1] - FIFTHS[0]) * b / max(bins.max(), 1)))
    return cats, colours


def overview_pages(zip_role, zip_cell, zip_sales, ctx):
    ests = sorted(zip_cell["EST_SOURCE"].unique())
    for role in sorted(zip_role["ROLE"].unique()):
        z = zip_role[zip_role["ROLE"] == role].copy()
        z["band"] = z["N_ROLE_HOLDERS"].map(lambda n: "4+" if n >= 4 else str(int(n)))
        cats = {b: list(g["ZIP_CODE"]) for b, g in z.groupby("band")}
        colours = {b: look.count_fill(n, "clean") for b, n in HOLDER_BANDS.items()}
        two_firms = int((z["N_ROLE_FIRMS"] > 1).sum())
        yield category_page(f"{role}: kept holders with an in-role piece on the ZIP", cats, colours, ctx, ests,
                            [f"Today's territories as they are.  2 or more is an overlap.  ZIPs held by both firms: {two_firms:,}."])
    for est in ests:
        c = zip_cell[zip_cell["EST_SOURCE"] == est].copy()
        c["status"] = "kept, in role"
        for col, label in (("OPP_OUT_OF_ROLE", "kept, out of role"), ("OPP_NO_HOLDER", "no holder today"),
                           ("OPP_OPEN", "open (holder not kept)")):
            c.loc[c[col] > 0, "status"] = label
        cats = {s: list(g["ZIP_CODE"]) for s, g in c.groupby("status")}
        c1 = (f"{est} estimate of the main map, {money(c['OPP'].sum())}: open {money(c['OPP_OPEN'].sum())}, no holder "
              f"{money(c['OPP_NO_HOLDER'].sum())}, kept out of role {money(c['OPP_OUT_OF_ROLE'].sum())}.")
        yield category_page(f"Main-map opportunity by holder status ({est} estimate)", cats, STATUS_COLOURS, ctx, [est],
                            [c1, "A ZIP takes the first of open, no holder, out of role that it has any opportunity in."])
    for est in ests:
        cats, colours = fifths(zip_cell[zip_cell["EST_SOURCE"] == est], "OPP", look.CLEAN_CMAP, ctx.polys)
        yield category_page(f"Main-map opportunity by ZIP ({est} estimate)", cats, colours, ctx, [est],
                            ["Fifths of the ZIPs with opportunity.  The two estimates are of one market: never add them."])
    for src in sorted(zip_sales["SOURCE"].unique()):
        cats, colours = fifths(zip_sales[zip_sales["SOURCE"] == src], "SALES", "Greens", ctx.polys)
        yield category_page(f"Main-map sales by ZIP ({src})", cats, colours, ctx, [src],
                            ["Fifths of the ZIPs with sales.  The sale's ZIP as the sales table has it."])


def wholesaler_page(src, name, p, w, ctx, zip_role):
    role = p["ROLE"].iloc[0]
    polys = ctx.polys
    inr = p[p["PIECE_IN_ROLE"] == 1].drop_duplicates("ZIP_CODE")
    out = p[(p["PIECE_IN_ROLE"] == 0) & ~p["ZIP_CODE"].isin(inr["ZIP_CODE"])].drop_duplicates("ZIP_CODE")
    sole, shared = inr[inr["N_ROLE_HOLDERS"].fillna(0) <= 1], inr[inr["N_ROLE_HOLDERS"].fillna(0) > 1]
    mine = [polys[z] for z in p["ZIP_CODE"].unique() if z in polys]
    box = None
    if mine:
        x0, y0, x1, y1 = shapely.total_bounds(mine)
        pad = max(x1 - x0, y1 - y0) * 0.08 + 20000
        box = (x0 - pad, y0 - pad, x1 + pad, y1 + pad)
    codes = str(w["ASSIGNED_STATES"].iloc[0] or "").split()
    others = [polys[z] for z in zip_role.loc[zip_role["ROLE"] == role, "ZIP_CODE"] if z in polys]

    def shapes(d):
        return [polys[z] for z in d["ZIP_CODE"] if z in polys]

    def paint(ax, box, fs):
        fill(ax, others, box, facecolor=OTHERS, edgecolor="none", zorder=1)
        fill(ax, shapes(sole), box, facecolor="#2171b5", edgecolor="#2171b5", linewidth=0.25, zorder=2)
        fill(ax, shapes(shared), box, facecolor="#fd8d3c", edgecolor="#fd8d3c", linewidth=0.25, zorder=2)
        fill(ax, shapes(out), box, facecolor="none", edgecolor="#555", hatch="////", linewidth=0.2, zorder=2)
        fill(ax, [ctx.st[c] for c in codes if c in ctx.st], box, facecolor="none", edgecolor="black", linewidth=1.6, zorder=5)

    handles = [Patch(facecolor="#2171b5", label=f"in role, sole kept {role} holder ({len(sole):,} ZIPs)"),
               Patch(facecolor="#fd8d3c", label=f"in role, shared with another kept {role} ({len(shared):,})"),
               Patch(facecolor="white", edgecolor="#555", hatch="////", label=f"out of role only ({len(out):,})"),
               Patch(facecolor=OTHERS, label=f"other kept {role} holders' ZIPs"),
               Patch(facecolor="white", edgecolor="black", linewidth=1.6, label="assigned states")]
    lines = [f"Role {role}", f"ZIPs held today {int(w['N_ZIPS'].iloc[0]):,}, in role {int(w['N_ZIPS_IN_ROLE'].iloc[0]):,}",
             f"States {w['STATES'].iloc[0] or '-'}", f"Assigned states {' '.join(codes) or '-'}", ""]
    for _, r in w.sort_values("EST_SOURCE").iterrows():
        lines += [f"Opportunity, {r['EST_SOURCE']} estimate", f"  on their pieces  {money(r['OPP_MAIN'])}",
                  f"  in role          {money(r['OPP_IN_ROLE'])}", f"  out of role      {money(r['OPP_OUT_OF_ROLE'])}",
                  f"  shared (overlap) {money(r['OPP_OVERLAP'])}"]
        if pd.notna(r["OPP_IN_ASSIGNED_STATES"]):
            lines.append(f"  in assigned st.  {money(r['OPP_IN_ASSIGNED_STATES'])}")
        if pd.notna(r["SALES_MAIN"]):
            pct = 100 * r["SALES_MAIN_IN_OWN_ZIPS"] / r["SALES_MAIN"] if r["SALES_MAIN"] else 0
            lines += ["", f"Main-map sales {money(r['SALES_MAIN'])}", f"  in their own ZIPs {pct:.0f}%"]
        lines.append("")
    c1 = f"{name} ({src}), {role}: their pieces as they are today; no wholesaler's territory is changed."
    c2 = "Blue and orange: ZIPs in role; hatched: out of role only; black outline: assigned states; dashed: state lines."
    return render(f"{name} ({src}), {role}", ctx, paint, handles, metro_list(ctx, [src], n=60), [c1, c2, note([src])],
                  box=box, insets=[], side="\n".join(lines), legend_fs=8)


def wholesaler_pages(pieces, wsum, ctx, zip_role):
    keys = wsum[["SOURCE", "WHOLESALER"]].drop_duplicates().sort_values(["SOURCE", "WHOLESALER"])
    for src, name in keys.itertuples(index=False):
        p = pieces[(pieces["SOURCE"] == src) & (pieces["WHOLESALER"] == name)]
        w = wsum[(wsum["SOURCE"] == src) & (wsum["WHOLESALER"] == name)]
        if len(p):
            yield wholesaler_page(src, name, p, w, ctx, zip_role)


def write_pdf(path, figs):
    """The pages as pictures, one at a time (a vector page of 30,000 ZIPs runs to megabytes)."""
    n = 0
    with PdfPages(path) as pdf:
        for fig in figs:
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=DPI, facecolor="white")
            w, h = fig.get_size_inches()
            plt.close(fig)
            buf.seek(0)
            pic = plt.figure(figsize=(w, h), dpi=DPI)
            axis = pic.add_axes([0, 0, 1, 1])
            axis.imshow(plt.imread(buf, format="png"), interpolation="none", aspect="auto")
            axis.set_axis_off()
            pdf.savefig(pic, dpi=DPI)
            plt.close(pic)
            n += 1
    if n == 0:
        raise RuntimeError(f"{path}: no page to draw")
    return n

In [ ]:
# Run: the two PDFs, then onto the stage
import datetime
import itertools
import os
import tempfile

ctx = SimpleNamespace(polys=polys, st=st, gap=gap, areas=areas, weights=metro_weights(zip_cell, zip_cbsa))
run = datetime.datetime.now().strftime("%Y-%m-%d-%H%M")
out = tempfile.mkdtemp()
for fname, figs in (("td_maps_territories_and_overview.pdf", itertools.chain(territory_pages(terr, ctx),
                                                                              overview_pages(zip_role, zip_cell, zip_sales, ctx))),
                    ("td_maps_wholesalers.pdf", wholesaler_pages(pieces, wsum, ctx, zip_role))):
    path = os.path.join(out, fname)
    n = write_pdf(path, figs)
    session.file.put(path, f"{STAGE}/{run}/", auto_compress=False, overwrite=True)
    print(f"{STAGE}/{run}/{fname}: {n} pages")
session.sql(f"ALTER STAGE {DB}.td_maps REFRESH").collect()